# Package

In [1]:
import mlflow
import pandas as pd

# Import Leaderbord

In [2]:
import os
import json
import joblib
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

EXPERIMENT_NAME = "H2_test"
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
assert exp is not None, f"Experiment introuvable: {EXPERIMENT_NAME}"
exp_id = exp.experiment_id

runs = mlflow.search_runs(
    experiment_ids=[exp_id],
    order_by=["start_time DESC"]
)
runs[["run_id","tags.mlflow.runName","start_time"]].head()

,run_id,tags.mlflow.runName,start_time
0,05b979d95b8a4ddbbbd4ba074ce73999,AR_ALL,2026-02-26 11:46:34.456000+00:00
1,0545d3fb429f4cdbb7b239aedee68f37,AR_2020-end,2026-02-26 11:46:33.799000+00:00
2,980d0b7bca454d2889196c20206361da,AR_2009-2019,2026-02-26 11:46:33.270000+00:00
3,ff4f3b48719b4235b3978dc612bcfd85,AR_2000-2008,2026-02-26 11:46:32.710000+00:00
4,6f85d12974444969a662e9c8cca4cf85,AR_1990-1999,2026-02-26 11:46:32.106000+00:00


In [3]:
def download_all_artifacts(run_id: str, dst_dir: str = "mlruns_download"):
    os.makedirs(dst_dir, exist_ok=True)
    local_dir = mlflow.artifacts.download_artifacts(run_id=run_id, dst_path=dst_dir)
    return local_dir

# Exemple: prendre le run le plus récent
latest_run_id = runs.iloc[0]["run_id"]
local_path = download_all_artifacts(latest_run_id, dst_dir="mlruns_download/latest")
local_path

'd:\\Portofolio Data science\\Time Series\\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\\3_notebook\\ML Experiment\\mlruns_download\\latest\\'

In [4]:
import glob

def load_run_bundle(local_path: str):
    out = {}

    # --- leaderboard ---
    lb = glob.glob(os.path.join(local_path, "leaderboard", "*.csv"))
    out["leaderboard"] = pd.read_csv(lb[0]) if lb else None

    # --- partition summary ---
    ps = glob.glob(os.path.join(local_path, "partition_summary.csv"))
    out["partition_summary"] = pd.read_csv(ps[0]) if ps else None

    # --- data parquet (bkt par partition + full) ---
    data_files = glob.glob(os.path.join(local_path, "data", "*.parquet"))
    out["data_parquets"] = {os.path.basename(p): pd.read_parquet(p) for p in data_files}

    # --- modèles (joblib) ---
    model_files = glob.glob(os.path.join(local_path, "model", "*.joblib"))
    out["models"] = {os.path.basename(p): joblib.load(p) for p in model_files}

    return out

bundle = load_run_bundle(local_path)

bundle["leaderboard"].head() if bundle["leaderboard"] is not None else "no leaderboard"

,unique_id,model_label,model_name,partition,mae,coverage,width,n
0,UNRATE,LR,LR,1990-1999,0.421220,0.800000,1.419842,120
1,UNRATE,LR,LR,2000-2008,0.377789,0.648148,0.981396,108
2,UNRATE,LR,LR,2009-2019,0.597483,0.780303,1.902029,132
3,UNRATE,LR,LR,2020-end,1.776643,0.764706,6.250181,68
4,UNRATE,LR,LR,ALL,0.679970,0.750000,2.225355,428


In [5]:
def download_and_load_experiment(exp_id: str, dst_root="mlruns_download/exp"):
    runs = mlflow.search_runs(experiment_ids=[exp_id], order_by=["start_time DESC"])
    all_bundles = {}

    for _, r in runs.iterrows():
        run_id = r["run_id"]
        run_name = r.get("tags.mlflow.runName", run_id)

        local = download_all_artifacts(run_id, dst_dir=os.path.join(dst_root, run_name))
        all_bundles[run_name] = load_run_bundle(local)

    return runs, all_bundles

runs_df, bundles = download_and_load_experiment(exp_id)
list(bundles.keys())[:5]

['AR_ALL', 'AR_2020-end', 'AR_2009-2019', 'AR_2000-2008', 'AR_1990-1999']

# Analyse

In [6]:
import pandas as pd

def get_ar_lr_leaderboard(bundles: dict, source_run_key: str = "AR_ALL"):
    df = bundles[source_run_key]["leaderboard"].copy()

    # garder seulement AR et LR
    df = df[df["model_label"].astype(str).isin(["AR", "LR"])].copy()

    # ordre partitions propre
    order = ["1990-1999","2000-2008","2009-2019","2020-end","ALL"]
    df["partition"] = pd.Categorical(df["partition"].astype(str), categories=order, ordered=True)

    df = df.sort_values(["model_label","partition"]).reset_index(drop=True)

    return df[["unique_id","model_label","model_name","partition","mae","coverage","width","n"]]

leaderboard_ar_lr = get_ar_lr_leaderboard(bundles, source_run_key="AR_ALL")
leaderboard_ar_lr

,unique_id,model_label,model_name,partition,mae,coverage,width,n
0,UNRATE,AR,AR,1990-1999,0.494189,0.816667,1.763992,120
1,UNRATE,AR,AR,2000-2008,0.514651,0.685185,1.569703,108
2,UNRATE,AR,AR,2009-2019,0.763062,0.825758,3.086353,132
3,UNRATE,AR,AR,2020-end,2.312401,0.661765,9.135620,68
4,UNRATE,AR,AR,ALL,0.871151,0.761682,3.293990,428
5,UNRATE,LR,LR,1990-1999,0.421220,0.800000,1.419842,120
6,UNRATE,LR,LR,2000-2008,0.377789,0.648148,0.981396,108
7,UNRATE,LR,LR,2009-2019,0.597483,0.780303,1.902029,132
8,UNRATE,LR,LR,2020-end,1.776643,0.764706,6.250181,68
9,UNRATE,LR,LR,ALL,0.679970,0.750000,2.225355,428


# toutes date pour LR

In [7]:
import numpy as np
import pandas as pd

def build_daily_metrics_LR_from_bundle(bundles: dict, unique_id="UNRATE"):
    # récupérer tous les dfs bkt_LR_<partition>.parquet (peu importe le run_key)
    part_frames = {}
    for run_key, b in bundles.items():
        for fname, df in b.get("data_parquets", {}).items():
            if fname.startswith("bkt_LR_") and fname.endswith(".parquet") and "score_full" not in fname:
                part = fname[len("bkt_LR_"):-len(".parquet")]
                part_frames.setdefault(part, df)

    if not part_frames:
        raise ValueError("Aucun bkt_LR_<partition>.parquet trouvé dans le bundle.")

    df = pd.concat(list(part_frames.values()), ignore_index=True)

    # normaliser
    df["unique_id"] = df.get("unique_id", unique_id)
    df["ds"] = pd.to_datetime(df["ds"])
    if "partition" not in df.columns:
        df["partition"] = "UNKNOWN"

    # sécurité bornes
    lo = np.minimum(df["LR-lo-95"].astype(float), df["LR-hi-95"].astype(float))
    hi = np.maximum(df["LR-lo-95"].astype(float), df["LR-hi-95"].astype(float))

    y = df["y"].astype(float)
    yhat = df["LR"].astype(float)

    out = pd.DataFrame({
        "unique_id": df["unique_id"].astype(str),
        "ds": df["ds"],
        "partition": df["partition"].astype(str),
        "model_label": "LR",
        "model_name": "LR",
        "y_true": y,
        "y_hat": yhat,
        "lo_95": lo,
        "hi_95": hi,
    })

    out["abs_err"] = (out["y_true"] - out["y_hat"]).abs()
    out["covered"] = ((out["y_true"] >= out["lo_95"]) & (out["y_true"] <= out["hi_95"])).astype(int)
    out["width"] = (out["hi_95"] - out["lo_95"]).astype(float)

    out = out.sort_values(["ds"]).reset_index(drop=True)
    return out

lr_daily = build_daily_metrics_LR_from_bundle(bundles)
lr_daily.head()

,unique_id,ds,partition,model_label,model_name,y_true,y_hat,lo_95,hi_95,abs_err,covered,width
0,UNRATE,1990-01-01,1990-1999,LR,LR,0.0,0.293094,-0.075968,0.662156,0.293094,1,0.738124
1,UNRATE,1990-02-01,1990-1999,LR,LR,0.1,0.022359,-0.432703,0.477422,0.077641,1,0.910125
2,UNRATE,1990-03-01,1990-1999,LR,LR,0.2,0.003990,-0.797507,0.805487,0.196010,1,1.602994
3,UNRATE,1990-04-01,1990-1999,LR,LR,0.2,-0.188218,-0.849793,0.473358,0.388218,1,1.323151
4,UNRATE,1990-05-01,1990-1999,LR,LR,0.2,-0.144925,-0.715670,0.425821,0.344925,1,1.141491


In [8]:
import os
import numpy as np
import pandas as pd
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

def load_ar_daily_from_mlflow(
    experiment_name: str = "Baseline",
    run_name_prefer: str = "AR_ALL",
    dst_dir: str = "mlruns_download/ar_baseline",
    unique_id: str = "UNRATE",
):
    # 1) trouver le run (id)
    exp = mlflow.get_experiment_by_name(experiment_name)
    if exp is None:
        raise ValueError(f"Experiment introuvable: {experiment_name}")

    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], output_format="pandas")
    if runs.empty:
        raise ValueError(f"Aucun run trouvé dans l'experiment {experiment_name}")

    # choisir AR_ALL si existe sinon run le plus récent
    if "tags.mlflow.runName" in runs.columns and (runs["tags.mlflow.runName"] == run_name_prefer).any():
        run_id = runs.loc[runs["tags.mlflow.runName"] == run_name_prefer, "run_id"].iloc[0]
    else:
        runs = runs.sort_values("start_time", ascending=False)
        run_id = runs["run_id"].iloc[0]

    # 2) télécharger l'artifact parquet
    os.makedirs(dst_dir, exist_ok=True)
    local_dir = mlflow.artifacts.download_artifacts(run_id=run_id, dst_path=dst_dir)

    # chemin attendu: <local_dir>/data/bkt_score_full.parquet
    path = os.path.join(local_dir, "data", "bkt_score_full.parquet")
    if not os.path.exists(path):
        # fallback: chercher dans data/
        data_dir = os.path.join(local_dir, "data")
        if not os.path.isdir(data_dir):
            raise FileNotFoundError(f"Dossier artifacts introuvable: {data_dir}")
        cands = [f for f in os.listdir(data_dir) if f.endswith(".parquet")]
        raise FileNotFoundError(f"bkt_score_full.parquet introuvable. Dispo dans data/: {cands}")

    df = pd.read_parquet(path)

    # 3) normaliser + features métriques par date
    df = df.copy()
    if "unique_id" not in df.columns:
        df["unique_id"] = unique_id

    df["ds"] = pd.to_datetime(df["ds"])

    # sécurité lower/upper
    lo = df["lo_95"].astype(float)
    hi = df["hi_95"].astype(float)
    df["lo_95"] = np.minimum(lo, hi)
    df["hi_95"] = np.maximum(lo, hi)

    df["y_true"] = df["y_true"].astype(float)
    df["y_hat"]  = df["y_hat"].astype(float)

    df["abs_err"] = (df["y_true"] - df["y_hat"]).abs()
    df["covered"] = ((df["y_true"] >= df["lo_95"]) & (df["y_true"] <= df["hi_95"])).astype(int)
    df["width"]   = (df["hi_95"] - df["lo_95"]).astype(float)

    # format "comme LR"
    out = df[[
        "unique_id", "ds", "partition",
        "y_true", "y_hat", "lo_95", "hi_95",
        "abs_err", "covered", "width"
    ]].sort_values("ds").reset_index(drop=True)

    return out, run_id, path


# ---- usage ----
ar_daily, ar_run_id, ar_local_path = load_ar_daily_from_mlflow(
    experiment_name="Baseline",
    run_name_prefer="AR_ALL",  # si tu as un run AR_ALL
)
print("AR run_id:", ar_run_id)
print("Loaded:", ar_local_path)
ar_daily.head()

AR run_id: 4a85f69b86244df091ad640c3a96d467
Loaded: d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\mlruns_download\ar_baseline\data\bkt_score_full.parquet


,unique_id,ds,partition,y_true,y_hat,lo_95,hi_95,abs_err,covered,width
0,UNRATE,1990-01-01,1990-1999,0.0,0.038995,-0.896105,0.974096,0.038995,1,1.870201
1,UNRATE,1990-02-01,1990-1999,0.1,-0.158664,-0.963583,0.646255,0.258664,1,1.609838
2,UNRATE,1990-03-01,1990-1999,0.2,-0.338061,-1.205755,0.529634,0.538061,1,1.735389
3,UNRATE,1990-04-01,1990-1999,0.2,0.079964,-0.698902,0.858830,0.120036,1,1.557732
4,UNRATE,1990-05-01,1990-1999,0.2,-0.004353,-0.955797,0.947091,0.204353,1,1.902888


In [9]:
import pandas as pd

def combine_ar_lr_bkt(ar_bkt: pd.DataFrame, lr_bkt: pd.DataFrame):

    cols_needed = [
        "unique_id", "ds", "partition",
        "y_true", "y_hat",
        "abs_err", "covered", "width"
    ]

    ar = ar_bkt.copy()
    lr = lr_bkt.copy()

    ar["model_label"] = "AR"
    lr["model_label"] = "LR"

    ar = ar[cols_needed + ["model_label"]]
    lr = lr[cols_needed + ["model_label"]]

    df = pd.concat([ar, lr], ignore_index=True)

    df = df.sort_values(["ds", "model_label"]).reset_index(drop=True)

    return df


# ---- usage ----
combined_bkt = combine_ar_lr_bkt(ar_daily, lr_daily)
combined_bkt.head()

,unique_id,ds,partition,y_true,y_hat,abs_err,covered,width,model_label
0,UNRATE,1990-01-01,1990-1999,0.0,0.038995,0.038995,1,1.870201,AR
1,UNRATE,1990-01-01,1990-1999,0.0,0.293094,0.293094,1,0.738124,LR
2,UNRATE,1990-02-01,1990-1999,0.1,-0.158664,0.258664,1,1.609838,AR
3,UNRATE,1990-02-01,1990-1999,0.1,0.022359,0.077641,1,0.910125,LR
4,UNRATE,1990-03-01,1990-1999,0.2,-0.338061,0.538061,1,1.735389,AR


In [17]:
ar_daily

,unique_id,ds,partition,y_true,y_hat,lo_95,hi_95,abs_err,covered,width
0,UNRATE,1990-01-01,1990-1999,0.0,0.038995,-0.896105,0.974096,0.038995,1,1.870201
1,UNRATE,1990-02-01,1990-1999,0.1,-0.158664,-0.963583,0.646255,0.258664,1,1.609838
2,UNRATE,1990-03-01,1990-1999,0.2,-0.338061,-1.205755,0.529634,0.538061,1,1.735389
3,UNRATE,1990-04-01,1990-1999,0.2,0.079964,-0.698902,0.858830,0.120036,1,1.557732
4,UNRATE,1990-05-01,1990-1999,0.2,-0.004353,-0.955797,0.947091,0.204353,1,1.902888
...,...,...,...,...,...,...,...,...,...,...
423,UNRATE,2025-04-01,2020-end,0.3,0.086570,-0.495027,0.668166,0.213430,1,1.163194
424,UNRATE,2025-05-01,2020-end,0.2,0.059958,-0.528360,0.648276,0.140042,1,1.176636
425,UNRATE,2025-06-01,2020-end,0.0,0.085175,-1.116228,1.286579,0.085175,1,2.402807
426,UNRATE,2025-07-01,2020-end,0.0,0.129088,-0.858917,1.117093,0.129088,1,1.976010


# Analyse

In [10]:
import numpy as np
import pandas as pd

def build_summary_table(combined_bkt: pd.DataFrame):

    df = combined_bkt.copy()

    # Harmoniser partitions
    df["partition"] = df["partition"].replace({
        "2009-2019": "2008-2019",
        "2020-end": "2020-fin"
    })

    order = ["1990-1999","2000-2008","2008-2019","2020-fin"]

    # fonction formatage
    def fmt(mean, std):
        if np.isnan(std):
            return f"{mean:.4f}"
        return f"{mean:.4f} ({std:.3f})"

    rows = []

    for model in df["model_label"].unique():

        row = {"model": model}

        df_m = df[df["model_label"] == model]

        # Ensemble (ALL)
        mean_all = df_m["abs_err"].mean()
        std_all  = df_m["abs_err"].std()
        row["Ensemble"] = fmt(mean_all, std_all)

        # Par partition
        for part in order:
            df_p = df_m[df_m["partition"] == part]

            if len(df_p) == 0:
                row[part] = ""
                continue

            mean_p = df_p["abs_err"].mean()
            std_p  = df_p["abs_err"].std()

            row[part] = fmt(mean_p, std_p)

        rows.append(row)

    summary = pd.DataFrame(rows)

    # ordre colonnes
    summary = summary[["model","Ensemble"] + order]

    return summary


# ---- usage ----
summary_table = build_summary_table(combined_bkt)
summary_table

,model,Ensemble,1990-1999,2000-2008,2008-2019,2020-fin
0,AR,0.8712 (1.661),0.4942 (0.358),0.5147 (0.479),0.7631 (0.839),2.3124 (3.607)
1,LR,0.6800 (1.122),0.4212 (0.374),0.3778 (0.316),0.5975 (0.401),1.7766 (2.409)


# MAE analysis

In [11]:
import numpy as np
import pandas as pd
from math import sqrt, erf, isfinite
from typing import Iterable, Optional, Dict, List, Tuple

# ============================================================
# 1) DM test helpers
# ============================================================
def _phi(z: float) -> float:
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))

def dm_pvalue(loss_diff: np.ndarray, lags: int = 0) -> float:
    x = np.asarray(loss_diff, dtype=float)
    x = x[np.isfinite(x)]
    T = x.size
    if T < 3:
        return np.nan

    dbar = x.mean()
    gamma0 = np.dot(x - dbar, x - dbar) / T
    var = gamma0

    if lags > 0:
        for k in range(1, min(lags, T - 1) + 1):
            w = 1.0 - k / (lags + 1.0)
            cov = np.dot(x[k:] - dbar, x[:-k] - dbar) / T
            var += 2.0 * w * cov

    if var <= 0:
        return np.nan

    stat = dbar / sqrt(var / T)
    p = 2.0 * (1.0 - _phi(abs(stat)))
    return max(0.0, min(1.0, p))


# ============================================================
# 2) MAE pivot + DM p-values (vs best model per segment)
# ============================================================
def make_mae_dm_pivot(
    wide: pd.DataFrame,
    segments: List[Tuple[str, Optional[str], str]],
    *,
    methods: Optional[Iterable[str]] = None,
    include_overall: bool = True,
    overall_label: str = "Ensemble",
    min_obs: int = 20,
    round_digits: int = 4,
    add_dm: bool = True,
    dm_lags: int = 11,   # ex: h-1 si h=12
) -> pd.DataFrame:

    df = wide.copy()

    # accepte colonne date ou index datetime
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
        df = df.set_index("date")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("wide doit avoir un DatetimeIndex ou une colonne 'date'.")

    df = df.sort_index()

    if "true" not in df.columns:
        raise ValueError("wide doit contenir la colonne 'true'.")

    if methods is None:
        meths = [c for c in df.columns if c != "true"]
    else:
        meths = [m for m in methods if m in df.columns and m != "true"]

    if len(meths) == 0:
        return pd.DataFrame()

    full_start, full_end = df.index.min(), df.index.max()

    windows: List[Tuple[pd.Timestamp, pd.Timestamp, str]] = []
    if include_overall:
        windows.append((full_start, full_end, overall_label))

    for start, end, label in segments:
        s = pd.to_datetime(start, utc=True)
        e = pd.to_datetime(end, utc=True) if end is not None else full_end
        windows.append((s, e, label))

    rows = []

    for start, end, label in windows:
        sub = df.loc[start:end, ["true"] + meths].copy().dropna(subset=["true"])

        maes: Dict[str, float] = {}
        err_abs: Dict[str, pd.Series] = {}

        for m in meths:
            diffs = (sub["true"] - sub[m]).abs().dropna()
            err_abs[m] = diffs
            maes[m] = float(diffs.mean()) if diffs.shape[0] >= min_obs else np.nan

        finite_models = [m for m in meths if isfinite(maes.get(m, np.nan))]
        best_m = min(finite_models, key=lambda k: maes[k]) if finite_models else None

        for m in meths:
            mae_val = maes.get(m, np.nan)
            if not isfinite(mae_val):
                rows.append((m, label, np.nan))
                continue

            cell = f"{mae_val:.{round_digits}f}"

            # DM p-value vs best model in the segment
            if add_dm and best_m is not None and m != best_m:
                v1 = err_abs[m]
                v2 = err_abs[best_m]
                common = v1.index.intersection(v2.index)
                diff = (v1.loc[common] - v2.loc[common]).to_numpy()

                if diff.size >= min_obs:
                    pval = dm_pvalue(diff, lags=dm_lags)
                    if isfinite(pval):
                        cell = f"{cell} ({pval:.3f})"

            rows.append((m, label, cell))

    out = pd.DataFrame(rows, columns=["model", "period", "value"])
    pivot = out.pivot(index="model", columns="period", values="value")

    desired_cols = ([overall_label] if include_overall else []) + [lbl for _, _, lbl in segments]
    pivot = pivot.reindex(columns=desired_cols)
    pivot = pivot.reindex(index=meths)

    pivot.columns.name = "period"
    pivot.index.name = "model"
    return pivot


# ============================================================
# 3) combined_bkt -> wide (true, AR, LR)
# ============================================================
def combined_bkt_to_wide_true_pred(
    combined_bkt: pd.DataFrame,
    *,
    date_col: str = "ds",
    true_col: str = "y_true",
    pred_col: str = "y_hat",
    model_col: str = "model_label",
    methods: Iterable[str] = ("AR", "LR"),
) -> pd.DataFrame:
    """
    Entrée: combined_bkt au format long (une ligne par ds x modèle)
      colonnes attendues: ds, model_label, y_true, y_hat
    Sortie: wide index=ds, colonnes=true + méthodes (AR, LR)
    """
    df = combined_bkt.copy()
    df[date_col] = pd.to_datetime(df[date_col], utc=True, errors="coerce")
    df = df.dropna(subset=[date_col])

    # pivot prédictions
    pred_wide = (
        df[df[model_col].astype(str).isin(list(methods))]
        .pivot_table(index=date_col, columns=model_col, values=pred_col, aggfunc="last")
        .sort_index()
    )

    # true (identique pour les deux) -> on prend la première valeur dispo par date
    true_ser = (
        df.groupby(date_col)[true_col]
          .apply(lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan)
          .sort_index()
    )

    wide = pred_wide.copy()
    wide.insert(0, "true", true_ser)
    return wide


# ============================================================
# 4) CALL (AR vs LR) -> tableau MAE + (DM p-value)
# ============================================================
# NOTE: combined_bkt doit déjà exister dans ton notebook.
# Il doit contenir: ds, model_label, y_true, y_hat (au minimum).

segments = [
    ("1990-01-01", "1999-12-01", "1990-1999"),
    ("2000-01-01", "2008-12-01", "2000-2008"),
    ("2009-01-01", "2019-12-01", "2008-2019"),  # label comme ton tableau
    ("2020-01-01", None,         "2020-fin"),
]

wide = combined_bkt_to_wide_true_pred(
    combined_bkt,
    methods=("AR", "LR"),
)

table_mae_dm = make_mae_dm_pivot(
    wide=wide,
    segments=segments,
    methods=["AR", "LR"],
    include_overall=True,
    overall_label="Ensemble",
    min_obs=20,
    round_digits=4,
    add_dm=True,
    dm_lags=11,
)

table_mae_dm

period,Ensemble,1990-1999,2000-2008,2008-2019,2020-fin
model,,,,,
AR,0.8712 (0.043),0.4942 (0.298),0.5147 (0.151),0.7631 (0.283),2.3124 (0.200)
LR,0.6800,0.4212,0.3778,0.5975,1.7766


From a macroeconomic perspective, the first observation is that forecast errors increase over time, especially during crisis periods.

From 1990–2008, MAE levels are relatively low for both models, reflecting a more stable and predictable macroeconomic environment. After 2008, errors rise, consistent with higher volatility, financial instability, and structural adjustments in the labor market. The sharp increase in 2020 reflects a clear structural break linked to the Covid shock—an exogenous event that standard time-series models are not designed to anticipate.

Throughout all periods, LR maintains lower errors than AR. This suggests that unemployment dynamics are not purely autoregressive; incorporating macroeconomic fundamentals improves the model’s ability to adapt to changing economic regimes.

Finally, the key statistical question is whether these differences are significant. If formal tests (e.g., Diebold–Mariano) confirm significance, then the superiority of LR is not due to random variation but reflects a structurally better information set. If not, the gap may be economically meaningful but statistically indistinguishable given sample size and volatility.

## Coverage

In [12]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from typing import List, Tuple, Optional, Iterable

# =========================================================
# 1) Helper: pivot covered (0/1) AR vs LR sur ds
# =========================================================
def combined_bkt_to_wide_covered(
    combined_bkt: pd.DataFrame,
    methods: Iterable[str] = ("AR", "LR"),
    date_col: str = "ds",
    model_col: str = "model_label",
    covered_col: str = "covered",
) -> pd.DataFrame:
    df = combined_bkt.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])

    # garde seulement AR/LR
    df = df[df[model_col].astype(str).isin(list(methods))].copy()

    # pivot covered
    wide = (
        df.pivot_table(index=date_col, columns=model_col, values=covered_col, aggfunc="last")
          .sort_index()
    )
    return wide

# =========================================================
# 2) McNemar sur coverage (paired), par segment
# =========================================================
def mcnemar_pvalue_from_pairs(c1: pd.Series, c2: pd.Series, *, exact=False, correction=True) -> float:
    # alignement
    common = c1.index.intersection(c2.index)
    c1 = c1.loc[common].dropna().astype(int)
    c2 = c2.loc[common].dropna().astype(int)

    common = c1.index.intersection(c2.index)
    c1 = c1.loc[common]
    c2 = c2.loc[common]

    if len(c1) == 0:
        return np.nan

    # contingence
    both_1 = int(np.sum((c1 == 1) & (c2 == 1)))
    c1_1  = int(np.sum((c1 == 1) & (c2 == 0)))
    c2_1  = int(np.sum((c1 == 0) & (c2 == 1)))
    both_0 = int(np.sum((c1 == 0) & (c2 == 0)))

    table = [[both_1, c1_1],
             [c2_1,  both_0]]

    res = mcnemar(table, exact=exact, correction=correction)
    return float(res.pvalue)

def make_coverage_mcnemar_table(
    combined_bkt: pd.DataFrame,
    segments: List[Tuple[str, Optional[str], str]],
    *,
    methods: List[str] = ["AR", "LR"],
    include_overall: bool = True,
    overall_label: str = "Ensemble",
    min_obs: int = 30,
    round_digits: int = 3,
) -> pd.DataFrame:
    """
    Sortie: tableau pivot
      ligne = model
      colonnes = Ensemble + segments
      cellule = "coverage" pour le meilleur modèle, sinon "coverage (p)" où p = McNemar vs best.
    """

    # wide covered
    wide = combined_bkt_to_wide_covered(combined_bkt, methods=methods)
    if not isinstance(wide.index, pd.DatetimeIndex):
        raise ValueError("Index date invalide après pivot.")

    full_start, full_end = wide.index.min(), wide.index.max()

    windows = []
    if include_overall:
        windows.append((full_start, full_end, overall_label))
    for start, end, label in segments:
        s = pd.to_datetime(start)
        e = pd.to_datetime(end) if end is not None else full_end
        windows.append((s, e, label))

    rows = []

    for start, end, label in windows:
        sub = wide.loc[start:end, methods].copy()

        # coverage par modèle + séries
        cov = {m: float(sub[m].dropna().mean()) if sub[m].dropna().shape[0] >= min_obs else np.nan for m in methods}
        nobs = {m: int(sub[m].dropna().shape[0]) for m in methods}

        finite = [m for m in methods if np.isfinite(cov[m])]
        best = max(finite, key=lambda m: cov[m]) if finite else None

        for m in methods:
            if not np.isfinite(cov[m]) or best is None:
                rows.append((m, label, np.nan))
                continue

            cell = f"{cov[m]:.{round_digits}f}"

            # p-value McNemar vs best (pour les non-best)
            if m != best:
                # pvalue nécessite des paires (dates communes)
                c1 = sub[m].dropna()
                c2 = sub[best].dropna()
                common = c1.index.intersection(c2.index)
                if len(common) >= min_obs:
                    p = mcnemar_pvalue_from_pairs(c1.loc[common], c2.loc[common], exact=False, correction=True)
                    if np.isfinite(p):
                        cell = f"{cell} ({p:.3f})"
                    else:
                        cell = f"{cell} (NaN)"
                else:
                    cell = f"{cell} (NaN)"

            rows.append((m, label, cell))

    out = pd.DataFrame(rows, columns=["model", "period", "value"])
    pivot = out.pivot(index="model", columns="period", values="value")

    desired_cols = ([overall_label] if include_overall else []) + [lbl for _, _, lbl in segments]
    pivot = pivot.reindex(columns=desired_cols)
    pivot = pivot.reindex(index=methods)

    pivot.columns.name = "period"
    pivot.index.name = "model"
    return pivot

# =========================================================
# 3) APPEL (segments comme ton pipeline)
# =========================================================
segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2009-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2020-fin"),
]

coverage_table = make_coverage_mcnemar_table(
    combined_bkt=combined_bkt,
    segments=segments,
    methods=["AR", "LR"],
    include_overall=True,
    overall_label="Ensemble",
    min_obs=30,
    round_digits=3,
)

coverage_table

period,Ensemble,1990-1999,2000-2008,2008-2019,2020-fin
model,,,,,
AR,0.762,0.817,0.685,0.826,0.662 (0.046)
LR,0.750 (0.661),0.800 (0.850),0.648 (0.571),0.780 (0.239),0.765


# Width

In [13]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from typing import List, Tuple, Optional, Iterable

# =========================================================
# 1) Wide width depuis combined_bkt
# =========================================================
def combined_bkt_to_wide_width(
    combined_bkt: pd.DataFrame,
    methods=("AR", "LR"),
    date_col="ds",
    model_col="model_label",
    width_col="width",
) -> pd.DataFrame:
    df = combined_bkt.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])
    df = df[df[model_col].astype(str).isin(list(methods))].copy()

    wide = (
        df.pivot_table(index=date_col, columns=model_col, values=width_col, aggfunc="last")
          .sort_index()
    )
    return wide

# =========================================================
# 2) Paired t-test p-value entre deux modèles (sur mêmes dates)
# =========================================================
def paired_t_pvalue(w1: pd.Series, w2: pd.Series, min_obs=30) -> float:
    common = w1.index.intersection(w2.index)
    a = w1.loc[common].dropna()
    b = w2.loc[common].dropna()
    common2 = a.index.intersection(b.index)
    a = a.loc[common2].astype(float)
    b = b.loc[common2].astype(float)
    if len(a) < min_obs:
        return np.nan
    return float(ttest_rel(a, b).pvalue)

# =========================================================
# 3) Table width "comme DM"
#    best = width moyen le plus PETIT sur le segment
# =========================================================
def make_width_ttest_pivot(
    combined_bkt: pd.DataFrame,
    segments: List[Tuple[str, Optional[str], str]],
    *,
    methods: Iterable[str] = ("AR", "LR"),
    include_overall: bool = True,
    overall_label: str = "Ensemble",
    min_obs: int = 30,
    round_digits: int = 4,
    add_test: bool = True,
) -> pd.DataFrame:

    wide = combined_bkt_to_wide_width(combined_bkt, methods=methods)

    full_start, full_end = wide.index.min(), wide.index.max()
    windows = []
    if include_overall:
        windows.append((full_start, full_end, overall_label))
    for start, end, label in segments:
        s = pd.to_datetime(start)
        e = pd.to_datetime(end) if end is not None else full_end
        windows.append((s, e, label))

    rows = []

    for start, end, label in windows:
        sub = wide.loc[start:end, list(methods)].copy()

        means = {m: float(sub[m].dropna().mean()) if sub[m].dropna().shape[0] >= min_obs else np.nan
                 for m in methods}

        finite = [m for m in methods if np.isfinite(means.get(m, np.nan))]
        best = min(finite, key=lambda k: means[k]) if finite else None  # width le plus petit

        for m in methods:
            mean_val = means.get(m, np.nan)
            if not np.isfinite(mean_val) or best is None:
                rows.append((m, label, np.nan))
                continue

            cell = f"{mean_val:.{round_digits}f}"

            # p-value vs best (paired t-test)
            if add_test and m != best:
                p = paired_t_pvalue(sub[m], sub[best], min_obs=min_obs)
                if np.isfinite(p):
                    cell = f"{cell} ({p:.3f})"

            rows.append((m, label, cell))

    out = pd.DataFrame(rows, columns=["model", "period", "value"])
    pivot = out.pivot(index="model", columns="period", values="value")

    desired_cols = ([overall_label] if include_overall else []) + [lbl for _, _, lbl in segments]
    pivot = pivot.reindex(columns=desired_cols)
    pivot = pivot.reindex(index=list(methods))

    pivot.columns.name = "period"
    pivot.index.name = "model"
    return pivot

# =========================================================
# 4) APPEL
# =========================================================
segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2009-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2020-fin"),
]

width_table = make_width_ttest_pivot(
    combined_bkt=combined_bkt,
    segments=segments,
    methods=("AR","LR"),
    include_overall=True,
    overall_label="Ensemble",
    min_obs=30,
    round_digits=4,  # ajuste si tu veux 3
    add_test=True,
)

width_table

period,Ensemble,1990-1999,2000-2008,2008-2019,2020-fin
model,,,,,
AR,3.2940 (0.000),1.7640 (0.000),1.5697 (0.000),3.0864 (0.000),9.1356 (0.000)
LR,2.2254,1.4198,0.9814,1.9020,6.2502


In [19]:
combined_bkt

,unique_id,ds,partition,y_true,y_hat,abs_err,covered,width,model_label
0,UNRATE,1990-01-01,1990-1999,0.0,0.038995,0.038995,1,1.870201,AR
1,UNRATE,1990-01-01,1990-1999,0.0,0.293094,0.293094,1,0.738124,LR
2,UNRATE,1990-02-01,1990-1999,0.1,-0.158664,0.258664,1,1.609838,AR
3,UNRATE,1990-02-01,1990-1999,0.1,0.022359,0.077641,1,0.910125,LR
4,UNRATE,1990-03-01,1990-1999,0.2,-0.338061,0.538061,1,1.735389,AR
...,...,...,...,...,...,...,...,...,...
851,UNRATE,2025-06-01,2020-end,0.0,-0.187980,0.187980,1,2.407029,LR
852,UNRATE,2025-07-01,2020-end,0.0,0.129088,0.129088,1,1.976010,AR
853,UNRATE,2025-07-01,2020-end,0.0,0.549694,0.549694,1,2.511152,LR
854,UNRATE,2025-08-01,2020-end,0.1,0.079615,0.020385,1,2.060421,AR


Here is the redaction in the same style as the MAE section:

---

From a macroeconomic and statistical perspective, coverage and width evolve clearly across regimes.

First, coverage declines during turbulent periods. In stable decades (1990–2008), coverage is relatively higher for both models, reflecting more predictable unemployment dynamics. However, during crisis regimes—especially 2008 and 2020—coverage deteriorates, indicating that both models underestimate extreme movements. This is consistent with structural breaks and sudden volatility spikes that standard time-series frameworks struggle to capture.

Second, interval width increases significantly in crisis periods. The sharp expansion in 2020 reflects heightened macroeconomic uncertainty during the Covid shock. This widening is economically coherent: when volatility rises, uncertainty bands must expand.

Comparatively, LR generally achieves narrower intervals than AR while maintaining similar or slightly better coverage in stressed periods. This suggests more efficient uncertainty quantification when macroeconomic information is included.

Finally, the key statistical question is whether differences in coverage and width between AR and LR are significant. Formal tests (e.g., coverage proportion tests or interval score comparisons) are required to determine whether LR provides statistically superior interval calibration or whether observed differences reflect sampling variability.

# Export

In [18]:
from pathlib import Path
import json
import pandas as pd

OUT = Path("outputs/app")
OUT.mkdir(parents=True, exist_ok=True)

# 1) Parquet wide (ds, y_true, M, M-lo-95, M-hi-95)
df_wide.to_parquet(OUT / "bkt_h2_wide.parquet", index=False)

# 2) Meta hyperparams
with open(OUT / "meta_AR.json", "w", encoding="utf-8") as f:
    json.dump(meta_ar, f, ensure_ascii=False, indent=2)

with open(OUT / "meta_LR.json", "w", encoding="utf-8") as f:
    json.dump(meta_lr, f, ensure_ascii=False, indent=2)

NameError: name 'df_wide' is not defined